In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
#Prepare a sample file text file for the demo
sample_text= """
Langchain is an open-source framework for building applications with large language models(LLMS)
It allows developers to combine LLMs with external data sources, apis and custom logic.
Retrieval-Augmented Generation(RAG) is a technique that combines information retrieval and LLMS
to answer questions with up-to-date, domain-specific, or long-tail knowledge.
"""

with open("sample.txt","w") as f:
    f.write(sample_text)


In [3]:
#Load Document

from langchain_community.document_loaders import TextLoader

loader= TextLoader("sample.txt")
docs= loader.load()
print("Loaded docs:", docs)

Loaded docs: [Document(metadata={'source': 'sample.txt'}, page_content='\nLangchain is an open-source framework for building applications with large language models(LLMS)\nIt allows developers to combine LLMs with external data sources, apis and custom logic.\nRetrieval-Augmented Generation(RAG) is a technique that combines information retrieval and LLMS\nto answer questions with up-to-date, domain-specific, or long-tail knowledge.\n')]


In [4]:
#Chunk the document

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)

chunks= splitter.split_documents(docs)
print(f"CHunks: {len(chunks)}")

for i,c in enumerate(chunks):
    print(f"Chunk {i+1}: {c.page_content}")

CHunks: 11
Chunk 1: Langchain is an open-source framework for
Chunk 2: for building applications with large language
Chunk 3: language models(LLMS)
Chunk 4: It allows developers to combine LLMs with
Chunk 5: LLMs with external data sources, apis and custom
Chunk 6: custom logic.
Chunk 7: Retrieval-Augmented Generation(RAG) is a
Chunk 8: is a technique that combines information
Chunk 9: retrieval and LLMS
Chunk 10: to answer questions with up-to-date,
Chunk 11: domain-specific, or long-tail knowledge.


In [5]:
# Generate Embeddings – Option 1: OpenAI

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # or "text-embedding-ada-002"
chunk_texts = [c.page_content for c in chunks]
vectors = embeddings.embed_documents(chunk_texts)
print(f"OpenAI Vector shape: {len(vectors)} chunks × {len(vectors[0])} dims")
print("First chunk vector (first 10 dims):", vectors[0][:10])

OpenAI Vector shape: 11 chunks × 1536 dims
First chunk vector (first 10 dims): [-0.03108166716992855, -0.015306745655834675, 0.05675330385565758, -0.053580112755298615, 0.03836439922451973, -0.03102964721620083, -0.012829315848648548, -0.0040022521279752254, -0.021614113822579384, -0.03446293622255325]


In [10]:
#Generate Embeddings – Option 2: HuggingFace (Free/Local)
!pip install sentence-transformers


from langchain_community.embeddings import HuggingFaceEmbeddings

hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")  # Small, fast model
hf_vectors = hf_embeddings.embed_documents(chunk_texts)
print(f"HuggingFace Vector shape: {len(hf_vectors)} chunks × {len(hf_vectors[0])} dims")
print("First chunk vector (first 10 dims):", hf_vectors[0][:10])


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 0.2.2 requires langchain-core!=0.3.0,!=0.3.1,!=0.3.10,!=0.3.11,!=0.3.12,!=0.3.13,!=0.3.14,!=0.3.2,!=0.3.3,!=0.3.4,!=0.3.5,!=0.3.6,!=0.3.7,!=0.3.8,!=0.3.9,<0.4.0,>=0.2.43, but you have langchain-core 1.2.17 which is incompatible.
c:\Users\adira\anaconda3\envs\langchainenvjan17\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   -------- ------------------------------- 2.4/10.7 MB 12.2 MB/s eta 0:00:01
   ------------------- -------------------- 5.2/10.7 MB 12.7 MB/s eta 0:00:01
   ------------------------------- -------- 8.4/10.7 MB 13.7 MB/s eta 0:00:01
   ---------------------------------------- 10.7/10.7 MB 12.8 MB/s  0:00:00
   ---------------------------------------- 0.0/612.9 kB ? eta -:--:--
   ---------------------------------------- 612.9/612.9 kB 11.3 MB/s  0:00:00
   ---------------------------------------- 0.0/3.6 MB ? eta -:--:--
   ---------------------------------- ----- 3.1/3.6 MB 16.9 MB/s eta 0:00:01
   ---------------------------------------- 3.6/3.6 MB 15.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 14.5 MB/s  0:00:00
   ------------------------------------

c:\Users\adira\anaconda3\envs\langchainenvjan17\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\adira\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4375.39it/s]
BertModel LOA

HuggingFace Vector shape: 11 chunks × 384 dims
First chunk vector (first 10 dims): [-0.05321227014064789, -0.038838885724544525, -0.02784501388669014, -0.06863018870353699, -1.4004986951476894e-05, -0.00907959509640932, -0.02428101934492588, 0.06700144708156586, 0.05679333955049515, -0.03288707882165909]


In [11]:
#Embed a Query for Retrieval
query = "What is RAG in LangChain?"
query_vec = embeddings.embed_query(query)  # For OpenAI
#hf_query_vec = hf_embeddings.embed_query(query)  # For HuggingFace

print("Query embedding (OpenAI):", query_vec[:10])
#print("Query embedding (HF):", hf_query_vec[:10])

Query embedding (OpenAI): [0.016138069331645966, 0.020839925855398178, 0.0007439805776812136, 0.030053935945034027, 0.004078554920852184, -0.020162424072623253, -0.00720860855653882, -0.015609619207680225, -0.0066361203789711, -0.033712439239025116]


In [12]:
#Embed a Query for Retrieval
query = "What is RAG in LangChain?"
#query_vec = embeddings.embed_query(query)  # For OpenAI
hf_query_vec = hf_embeddings.embed_query(query)  # For HuggingFace

#print("Query embedding (OpenAI):", query_vec[:10])
print("Query embedding (HF):", hf_query_vec[:10])



Query embedding (HF): [-0.05358980596065521, 0.07530609518289566, 0.04431810602545738, -0.01331262569874525, -0.09721583873033524, 0.04635937511920929, 0.05921627953648567, 0.042014990001916885, 0.019781265407800674, -0.04648131877183914]


In [13]:
#Compute Similarity (Cosine) between query and chunks

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Cosine similarity between query and all chunk vectors

sims = cosine_similarity([query_vec], vectors)[0]
print("open ai cosine similarities: ", sims)


top_idx= int(np.argmax(sims))

print("Most relevant chunk (Openai):", chunks[top_idx].page_content)

open ai cosine similarities:  [0.44859752 0.27775698 0.23679103 0.24324999 0.17723016 0.19326323
 0.43231952 0.1342658  0.22066113 0.06974758 0.156868  ]
Most relevant chunk (Openai): Langchain is an open-source framework for


In [14]:
hf_sims = cosine_similarity([hf_query_vec], hf_vectors)[0]
print("HuggingFace Cosine similarities:", hf_sims)

HuggingFace Cosine similarities: [ 0.41203323  0.14212615  0.12862927  0.11449242 -0.06132904  0.03322068
  0.46011665  0.11888321  0.11790224  0.00540383  0.07166499]
